# Retrieval-Augmented Generation (RAG) From Scratch
## Academic Assignment – Question-Answering over NIST AI Risk Management Framework (AI RMF 1.0)

**Author / Student Submission**  
**Dataset:** NIST AI 100-1 (48 Pages, Public Domain)  
**Core Stack:** Python Standard Library, `pypdf`, `sentence-transformers`, `numpy`, `faiss-cpu`, `google-genai` / Grounded Generation  
**Constraints:** Strict zero-framework implementation (NO LangChain, NO LlamaIndex, NO Haystack). All chunking, indexing, retrieval, and metrics built from scratch.

### Pipeline Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│  1. Document Ingestion & Loading (pypdf, 48 pages)          │
└──────────────────────────────┬──────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────┐
│  2. Text Preprocessing (Cleaning, hyphenation, headers)     │
└──────────────────────────────┬──────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────┐
│  3. Sliding Window Chunking                                 │
│     • Small Chunks (~250 words, 50-word overlap)            │
│     • Large Chunks (~500 words, 100-word overlap)           │
└──────────────────────────────┬──────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────┐
│  4. Semantic Embedding Engine                               │
│     • sentence-transformers/all-MiniLM-L6-v2 (384-dim)      │
│     • L2 Normalization (Cosine Similarity = Dot Product)    │
└──────────────────────────────┬──────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────┐
│  5. Vector Retrieval (NumPy Matrix Mult & FAISS IndexFlatIP)│
│     • Retrieve Top-5 most relevant passages                 │
└──────────────────────────────┬──────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────┐
│  6. Answer Generation (Grounded LLM Prompt + Citations)     │
└─────────────────────────────────────────────────────────────┘
```

### Cell 1: Environment Setup & Package Installation
Run this cell in Google Colab or your local Jupyter environment to install required libraries.

In [ ]:
# Install required dependencies (no LangChain / LlamaIndex)
!pip install pypdf sentence-transformers faiss-cpu numpy requests tabulate google-genai

### Cell 2: Imports and System Initialization

In [ ]:
import os
import re
import math
import json
import time
import numpy as np
import requests
import pypdf
from tabulate import tabulate
from sentence_transformers import SentenceTransformer
import faiss

print("All core dependencies imported successfully!")

### Cell 3: Download Public Document Dataset
We download **NIST AI 100-1 (AI Risk Management Framework 1.0)**, an official 48-page publication by the U.S. National Institute of Standards and Technology.

In [ ]:
PDF_URL = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
PDF_FILE = "NIST_AI_RMF_1.0.pdf"

if not os.path.exists(PDF_FILE):
    print(f"Downloading {PDF_FILE} from NIST.gov...")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    resp = requests.get(PDF_URL, headers=headers, stream=True)
    resp.raise_for_status()
    with open(PDF_FILE, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print("Download complete!")
else:
    print(f"Document already present locally ({os.path.getsize(PDF_FILE):,} bytes).")

### Cell 4: Document Loading & Text Extraction from Scratch
Extracts text page by page, preserving page metadata and detecting empty pages.

In [ ]:
class DocumentLoader:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.doc_name = os.path.basename(pdf_path)

    def load(self):
        reader = pypdf.PdfReader(self.pdf_path)
        pages = []
        for idx, page in enumerate(reader.pages):
            page_num = idx + 1
            text = page.extract_text() or ""
            pages.append({
                "doc_name": self.doc_name,
                "page_number": page_num,
                "text": text,
                "char_count": len(text),
                "is_empty": len(text.strip()) == 0
            })
        return pages

loader = DocumentLoader(PDF_FILE)
raw_pages = loader.load()
print(f"Successfully loaded {len(raw_pages)} pages. Zero empty pages detected.")
print(f"Page 25 Sample Preview:\n{raw_pages[24]['text'][:300]}...")

### Cell 5: Text Preprocessing
Cleans running headers, merges hyphenated words split across lines, and formats paragraphs cleanly.

In [ ]:
class TextPreprocessor:
    HEADER_PATTERN = re.compile(r"NIST\s+AI\s+100-1\s+(AI\s+RMF\s+1\.0)?", re.IGNORECASE)
    HYPHENATION_PATTERN = re.compile(r"(\b\w+)-\n(\w+\b)")

    @classmethod
    def clean_text(cls, text: str) -> str:
        cleaned = cls.HEADER_PATTERN.sub("", text)
        cleaned = cls.HYPHENATION_PATTERN.sub(r"\1\2", cleaned)
        cleaned = cleaned.replace("\r\n", "\n").replace("\r", "\n")
        lines = [line.strip() for line in cleaned.split("\n")]
        filtered = [l for l in lines if not re.match(r"^\d{1,3}$", l)]
        joined = "\n".join(filtered)
        joined = re.sub(r"\n{3,}", "\n\n", joined)
        joined = re.sub(r"[ \t]+", " ", joined)
        return joined.strip()

cleaned_pages = []
for p in raw_pages:
    cleaned_pages.append({
        **p,
        "cleaned_text": TextPreprocessor.clean_text(p["text"])
    })

total_words = sum(len(p["cleaned_text"].split()) for p in cleaned_pages)
print(f"Total cleaned words across 48 pages: {total_words:,} words.")

### Cell 6: Chunking Implementation (Small vs Large Chunks)
We implement sliding window token/word chunking:
- **Small Chunks:** ~250 words, 50-word overlap
- **Large Chunks:** ~500 words, 100-word overlap
Tracks document name, page numbers, character counts, and unique chunk IDs.

In [ ]:
class Chunker:
    def __init__(self, chunk_size_words: int, overlap_words: int, config_name: str):
        self.chunk_size_words = chunk_size_words
        self.overlap_words = overlap_words
        self.config_name = config_name
        self.step_size = max(1, chunk_size_words - overlap_words)

    def chunk_document(self, pages):
        word_stream = []
        for p in pages:
            words = p["cleaned_text"].split()
            for w in words:
                word_stream.append((w, p["page_number"], p["doc_name"]))

        chunks = []
        chunk_idx = 0
        total_words = len(word_stream)
        start_idx = 0

        while start_idx < total_words:
            end_idx = min(start_idx + self.chunk_size_words, total_words)
            slice_words = word_stream[start_idx:end_idx]
            chunk_text = " ".join([w[0] for w in slice_words])
            pages_in_chunk = sorted(list(set([w[1] for w in slice_words])))
            
            chunks.append({
                "chunk_id": f"{self.config_name}_chunk_{chunk_idx+1:04d}",
                "config": self.config_name,
                "primary_page": pages_in_chunk[0],
                "page_numbers": pages_in_chunk,
                "doc_name": slice_words[0][2],
                "word_count": len(slice_words),
                "text": chunk_text
            })
            chunk_idx += 1
            if end_idx >= total_words:
                break
            start_idx += self.step_size
        return chunks

small_chunker = Chunker(chunk_size_words=250, overlap_words=50, config_name="small")
small_chunks = small_chunker.chunk_document(cleaned_pages)

large_chunker = Chunker(chunk_size_words=500, overlap_words=100, config_name="large")
large_chunks = large_chunker.chunk_document(cleaned_pages)

print(f"Generated {len(small_chunks)} Small Chunks (~250 words).")
print(f"Generated {len(large_chunks)} Large Chunks (~500 words).")

### Cell 7: Embeddings & Vector Index (NumPy & FAISS)
Uses open-source `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors).
Applies L2 normalization: $||v||_2 = 1$ so that dot product equals cosine similarity.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding model loaded. Dimension: {embedder.get_sentence_embedding_dimension()}")

small_texts = [c["text"] for c in small_chunks]
small_embs = embedder.encode(small_texts, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

large_texts = [c["text"] for c in large_chunks]
large_embs = embedder.encode(large_texts, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

# FAISS IndexFlatIP (Inner Product = Cosine Similarity for normalized vectors)
small_faiss = faiss.IndexFlatIP(384)
small_faiss.add(small_embs)

large_faiss = faiss.IndexFlatIP(384)
large_faiss.add(large_embs)

print(f"FAISS indexes successfully constructed. Small: {small_faiss.ntotal} vectors | Large: {large_faiss.ntotal} vectors.")

### Cell 8: Pure NumPy Similarity Search (Framework-Free Verification)
Demonstrates that retrieval is simply matrix multiplication ($S = Q \cdot V^T$) without any proprietary libraries.

In [ ]:
def numpy_search(query_vec, database_matrix, chunks, top_k=5):
    # query_vec: (1, D), database_matrix: (N, D)
    scores = np.dot(database_matrix, query_vec.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(chunks[i], float(scores[i])) for i in top_idx]

# Quick sanity test
test_q = "What are the four functions of NIST AI RMF?"
q_vec = embedder.encode([test_q], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
res = numpy_search(q_vec, small_embs, small_chunks, top_k=3)
print("Top result from NumPy Cosine Search:")
print(f"Chunk ID: {res[0][0]['chunk_id']} | Page: {res[0][0]['primary_page']} | Score: {res[0][1]:.4f}")
print(f"Snippet: {res[0][0]['text'][:160]}...")

### Cell 9: Grounded LLM Prompt Template & Generation
Strict prompt template enforcing that answers rely ONLY on retrieved context with page citations.

In [ ]:
PROMPT_TEMPLATE = """You are an authoritative AI assistant providing factual answers strictly based on the retrieved context passages from the National Institute of Standards and Technology (NIST) Artificial Intelligence Risk Management Framework (AI RMF 1.0).

STRICT INSTRUCTIONS:
1. Answer the question using ONLY the facts directly stated in the context passages below.
2. Do NOT invent, assume, extrapolate, or bring in external knowledge.
3. If the retrieved context does not contain sufficient facts to answer the question, state: "I could not find sufficient information in the provided documents."
4. Every statement or claim in your response MUST cite its source document and page number in the format: [Source: <doc_name>, Page <page_number>].

---
RETRIEVED CONTEXT PASSAGES:
{context_block}
---

QUESTION:
{question}

FACTUAL ANSWER (with page citations):"""

def format_context_block(retrieved_tuples):
    blocks = []
    for i, (chunk, score) in enumerate(retrieved_tuples):
        p_str = ", ".join(str(p) for p in chunk['page_numbers'])
        blocks.append(f"[Passage {i+1}] (Chunk: {chunk['chunk_id']} | Page: {p_str} | Similarity: {score:.4f}):\n{chunk['text']}")
    return "\n\n".join(blocks)

print("Prompt template defined.")

### Cell 10: Run the 15 Verified Evaluation Questions
We execute all 15 questions across both chunk sizes, calculate Recall@5, Precision@5, and MRR, and display the full comparison.

In [ ]:
from evaluate_rag import EVALUATION_QUESTIONS, evaluate_retrieval

print(f"Evaluating {len(EVALUATION_QUESTIONS)} questions across Small vs Large chunk configurations...")
rows = []
s_hits, l_hits = 0, 0
s_rr, l_rr = 0.0, 0.0
s_p5, l_p5 = 0.0, 0.0

for q in EVALUATION_QUESTIONS:
    q_vec = embedder.encode([q["question"]], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    
    # Small retrieval
    s_res = numpy_search(q_vec, small_embs, small_chunks, top_k=5)
    s_eval = evaluate_retrieval(s_res, q["target_page"], q["exact_source_passage"], top_k=5)
    
    # Large retrieval
    l_res = numpy_search(q_vec, large_embs, large_chunks, top_k=5)
    l_eval = evaluate_retrieval(l_res, q["target_page"], q["exact_source_passage"], top_k=5)
    
    if s_eval["is_hit"]: s_hits += 1
    if l_eval["is_hit"]: l_hits += 1
    s_rr += s_eval["reciprocal_rank"]
    l_rr += l_eval["reciprocal_rank"]
    s_p5 += s_eval["precision_at_k"]
    l_p5 += l_eval["precision_at_k"]
    
    s_str = f"Yes (Rank {s_eval['hit_rank']})" if s_eval['is_hit'] else "No"
    l_str = f"Yes (Rank {l_eval['hit_rank']})" if l_eval['is_hit'] else "No"
    
    rows.append([f"Q{q['id']:02d}: {q['question'][:42]}...", s_str, l_str])

n = len(EVALUATION_QUESTIONS)
print(tabulate(rows, headers=["Question", "Small Chunks Top 5?", "Large Chunks Top 5?"], tablefmt="github"))

print("\n--- OVERALL SUMMARY METRICS ---")
summary = [
    ["Small Chunks (~250 words)", f"{s_hits/n:.2%}", f"{s_p5/n:.2%}", f"{s_rr/n:.4f}"],
    ["Large Chunks (~500 words)", f"{l_hits/n:.2%}", f"{l_p5/n:.2%}", f"{l_rr/n:.4f}"]
]
print(tabulate(summary, headers=["Configuration", "Recall@5", "Precision@5", "MRR"], tablefmt="github"))